In [2]:
# IMPORTS
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Module imports
from prp.task_generator import generate_task_patterns, generate_fixed_task_set
from prp.nn_wrapper import TaskNetworkWrapper
from prp.prp_simulator import sweep_soa, run_prp_trial
from prp.training_utils import train_with_optional_multitask, train_with_control_config 
from prp.threshold_utils import optimize_lca_threshold, choose_onset_policy, optimize_reward_rate_threshold


# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)


In [2]:
# Reload module when making live edits
import importlib
import prp.prp_simulator as prp_simulator
importlib.reload(prp_simulator)

<module 'prp.prp_simulator' from '/home/acesmeci/Code/prp_model/prp/prp_simulator.py'>

In [ ]:
# Train the model. Ideally use python scripts/train_model.py in terminal instead.

# Generate fixed task set A–E for training
inp, task, target, _ = generate_fixed_task_set(
    N_pathways=3,
    N_features=3,
    samples_per_task=100,
)

net = TaskNetworkWrapper(hidden_size=100)
net.train_online(inp, task, target, stop_loss=0.001) # Paper stop_loss = 0.001

# Save model after training
torch.save(net.model.state_dict(), "../output/trained_model_0001.pth")
print("✅ Model saved to output/trained_model.pth")


In [ ]:
# New model loading cell
from prp.nn_wrapper import TaskNetworkWrapper
import torch

# Match exactly what you used in scripts/train_model.py:
net = TaskNetworkWrapper(
    stim_input_dim = 3 * 3,    # N_pathways * N_features
    task_input_dim = 3 ** 2,   # N_pathways^2
    hidden_dim     = 100,
    output_dim     = 3 * 3,
    learning_rate  = 0.3,
    device         = "cpu"
)

# Now load
net.model.load_state_dict(torch.load("../output/trained_models/trained_model.pth"))
net.model.eval()
print("✅ Loaded 0.001 model, ready for inference.")


# L = 0.001 
net_2 = TaskNetworkWrapper(
    stim_input_dim = 3 * 3,    # N_pathways * N_features
    task_input_dim = 3 ** 2,   # N_pathways^2
    hidden_dim     = 100,
    output_dim     = 3 * 3,
    learning_rate  = 0.3,
    device         = "cpu"
)

# Now load
wrapper.model.load_state_dict(torch.load("../output/trained_model.pth"))
wrapper.model.eval()
print("✅ Loaded model, ready for inference.")

### New training (MATLAB-style)

In [3]:
import torch
from prp.nn_wrapper import TaskNetworkWrapper
from prp.training_set import generate_training_set_matlab_style  # or wherever you placed it

# 1) Build exhaustive, noise-free, same across tasks
X, T, Y, _ = generate_training_set_matlab_style()

# 2) Train
wrapper = TaskNetworkWrapper(
    stim_input_dim=9, task_input_dim=9, hidden_dim=100, output_dim=9,
    learning_rate=0.3, device="cpu"
)
wrapper.train_online(
    torch.tensor(X), torch.tensor(T), torch.tensor(Y),
    max_epochs=3000, stop_loss=1e-3, print_every=50
)

torch.save(wrapper.model.state_dict(), "../output/trained_models/trained_model_init01_all.pth")
print("✅ Model saved to output/trained_model_init005.pth")

# 3) Check Pearson AD/BE on task→hidden (same code you already have)
# Expectation: AD/BE should jump noticeably vs. the random/noisy regime.


Epoch 0000 | Loss: 0.0984 | Acc: 0.096
Epoch 0050 | Loss: 0.0748 | Acc: 0.578
Epoch 0100 | Loss: 0.0528 | Acc: 0.659
Epoch 0150 | Loss: 0.0426 | Acc: 0.681
Epoch 0200 | Loss: 0.0347 | Acc: 0.778
Epoch 0250 | Loss: 0.0304 | Acc: 0.948
Epoch 0300 | Loss: 0.0209 | Acc: 1.000
Epoch 0350 | Loss: 0.0105 | Acc: 1.000
Epoch 0400 | Loss: 0.0055 | Acc: 1.000
Epoch 0450 | Loss: 0.0034 | Acc: 1.000
Epoch 0500 | Loss: 0.0024 | Acc: 1.000
Epoch 0550 | Loss: 0.0019 | Acc: 1.000
Epoch 0600 | Loss: 0.0015 | Acc: 1.000
Epoch 0650 | Loss: 0.0013 | Acc: 1.000
Epoch 0700 | Loss: 0.0011 | Acc: 1.000
✅ Converged at epoch 0731 | Loss: 0.0010
✅ Model saved to output/trained_model_init005.pth


In [4]:
wrapper.model.eval()

TaskNetwork(
  (fc_input_hidden): Linear(in_features=9, out_features=100, bias=False)
  (fc_task_hidden): Linear(in_features=9, out_features=100, bias=False)
  (fc_hidden_output): Linear(in_features=100, out_features=9, bias=False)
  (fc_task_output): Linear(in_features=9, out_features=9, bias=False)
  (act): Sigmoid()
)

### Init-fixed training

In [3]:
# Construct the wrapper
from prp.nn_wrapper import TaskNetworkWrapper

wrapper = TaskNetworkWrapper(
    stim_input_dim = 3*3,
    task_input_dim = 3**2,
    hidden_dim     = 100,
    output_dim     = 3*3,
    learning_rate  = 0.3,
    # Sim-3 init/bias/decay:
    init_scale=0.1,
    init_task_scale=None,   # → 0.1, same as MATLAB for Sim-3
    bias_offset=-2.0,
    default_weight_decay=0.0,  # Sim-3 uses no L2
    device="cpu",
)
print("✅ Model constructed.")


✅ Model constructed.


In [4]:
# Generate training set & train
import torch
from prp.training_set import generate_training_set_matlab_style

# 1) Exhaustive, noise-free, same across tasks (A,B,D,E,C)
X, T, Y, _ = generate_training_set_matlab_style()

# 2) Train
wrapper.train_online(
    torch.tensor(X), torch.tensor(T), torch.tensor(Y),
    max_epochs=5000, stop_loss=1e-3, print_every=50
)

# 3) Save
#torch.save(wrapper.model.state_dict(), "../output/trained_models/trained_model_sim3_init01.pth")
print("✅ Saved.")


/home/acesmeci/code/prp_model/prp/nn_wrapper.py:105: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  total_loss += float(loss)


Epoch 0000 | Loss: 0.0986 | Acc: 0.178
Epoch 0050 | Loss: 0.0745 | Acc: 0.585
Epoch 0100 | Loss: 0.0526 | Acc: 0.674
Epoch 0150 | Loss: 0.0420 | Acc: 0.726
Epoch 0200 | Loss: 0.0346 | Acc: 0.711
Epoch 0250 | Loss: 0.0316 | Acc: 0.830
Epoch 0300 | Loss: 0.0258 | Acc: 0.963
Epoch 0350 | Loss: 0.0150 | Acc: 1.000
Epoch 0400 | Loss: 0.0072 | Acc: 1.000
Epoch 0450 | Loss: 0.0039 | Acc: 1.000
Epoch 0500 | Loss: 0.0026 | Acc: 1.000
Epoch 0550 | Loss: 0.0019 | Acc: 1.000
Epoch 0600 | Loss: 0.0015 | Acc: 1.000
Epoch 0650 | Loss: 0.0012 | Acc: 1.000
Epoch 0700 | Loss: 0.0010 | Acc: 1.000
✅ Converged at epoch 0713 | Loss: 0.0010
✅ Saved.


In [5]:
# Explore the stimulus set
# X: Input stimuli, T: Task cues, Y: Target outputs
from prp.training_set import generate_training_set_matlab_style
X, T, Y, _ = generate_training_set_matlab_style()
print(X.shape, T.shape, Y.shape)
print(T[:,8])

(135, 9) (135, 9) (135, 9)
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


### Load model

In [25]:
# Load the trained model
PATH = "../output/trained_models/trained_model_xavier_g1.pth"
wrapper.model.load_state_dict(torch.load("../output/trained_models/trained_model_init005.pth"))
wrapper.model.eval()

TaskNetwork(
  (fc_input_hidden): Linear(in_features=9, out_features=100, bias=False)
  (fc_task_hidden): Linear(in_features=9, out_features=100, bias=False)
  (fc_hidden_output): Linear(in_features=100, out_features=9, bias=False)
  (fc_task_output): Linear(in_features=9, out_features=9, bias=False)
  (act): Sigmoid()
)

### Compute Fixed Task-2 Threshold

In [ ]:
# === Notebook cell: compute fixed Task-B threshold ===

import numpy as np
import torch
from prp.task_generator   import generate_fixed_task_set
from prp.threshold_utils  import optimize_lca_threshold_dist
from prp.lca              import run_lca_avg

# 1) Get a batch of Task B examples
inp, tasks_sgl, train_sgl, meta = generate_fixed_task_set(seed=SEED)
mask_B = (meta["task_indices"] == "B")
input_B  = inp[mask_B]
task_B   = tasks_sgl[mask_B]

# 2) Pick one representative trial (or average across several)
input_b = input_B[0]
task_b  = task_B[0]

# 3) Run the network with no carry-over (pure single-task B)
output_series_th = net_2.integrate(
    torch.from_numpy(input_b[None,:]).float(),
    torch.from_numpy(task_b[None,:]).float(),
    persistence=0.0, # Look into this
)
output_np = np.stack([o.numpy() for o in output_series_th], axis=0)

# 4) Decode which output indices correspond to B
N_pathways, N_features = 3, 3
mat_B = task_b.reshape(N_pathways, N_pathways).T
in_B, out_B = np.argwhere(mat_B==1)[0]
idxs_b = list(range(out_B*N_features, (out_B+1)*N_features))
correct_b = np.argmax(input_b[in_B*N_features:(in_B+1)*N_features])

# 5) Fit LCA threshold once
thresholds = np.arange(0.0, 1.6, 0.1)
z_B, rr_B = optimize_lca_threshold_dist(
    output_np, idxs_b, correct_response_idx=correct_b,
    thresholds=thresholds, ITI=0.5, n_repeats=100
)
print("Fixed Task B threshold =", z_B)

In [ ]:
# PRP Simulator. Takes ~2 minutes to run with 10 trials per SOA.
# ~5 minutes with 50 trials per SOA.

def generate_trial_pair(prp_pair=("B","A"), N_pathways=3, N_features=3, seed=None):
    task_map = {'A': (0,0), 'B': (1,1), 'C': (2,2), 'D': (0,1), 'E': (1,0)}
    rng = np.random.RandomState(seed)

    def sample_single_task(task_name, shared_features=None):
        in_dim, out_dim = task_map[task_name]
        feats = shared_features if shared_features is not None \
                else rng.randint(0, N_features, size=N_pathways)

        stim = np.zeros(N_pathways*N_features, dtype=np.float32)
        for i in range(N_pathways):
            stim[i*N_features + feats[i]] = 1

        cue = np.zeros(N_pathways**2, dtype=np.float32)
        cue[in_dim*N_pathways + out_dim] = 1
        return stim, cue

    # (optional) use the SAME stimulus features for both tasks:
    feats = np.random.randint(0, N_features, size=N_pathways)
    stim1, cue1 = sample_single_task(prp_pair[0], shared_features=feats)
    stim2, cue2 = sample_single_task(prp_pair[1], shared_features=feats)
    return stim1, stim2, cue1, cue2


# ✅ Optional test calls
_ = generate_trial_pair(prp_pair=("B", "A"))
_ = generate_trial_pair(prp_pair=("C", "A"))


# ✅ Sweep for B → A (functionally dependent)
results_ba = sweep_soa(
    task_net=wrapper,
    trial_generator=lambda: generate_trial_pair(("B", "A")),
    soa_values=list(range(0, 9)),  # SOAs 0–8
    n_trials_per_soa=100,           # Optional: increase to 50 for final experiments
    #tau_net=0.2,
    #tau_task=0.2,
    persistence=0.8, # run with 0.9
    z_b_fixed=z_B,  # Use the fixed threshold from Task B
    verbose=False
)

# ✅ Sweep for C → A (functionally independent)
results_ca = sweep_soa(
    task_net=wrapper,
    trial_generator=lambda: generate_trial_pair(("C", "A")),
    soa_values=list(range(0, 9)), # changed dt = 0.05, so this should cover 50ms to 400ms
    n_trials_per_soa=100,
    #tau_net=0.2,
    #tau_task=0.2,
    persistence=0.8, # 0.9 for proper interference.
    z_b_fixed=z_B,
    verbose=False
)

In [ ]:
# Plot RTs and Error Rates

# RTs
plt.plot(results_ba["soa"], results_ba["rt_task1"], label="Task B RT (Dep)", marker='o')
plt.plot(results_ba["soa"], results_ba["rt_task2"], label="Task A RT (Dep)", marker='x', linestyle='--')

plt.plot(results_ca["soa"], results_ca["rt_task1"], label="Task C RT (Indep)", marker='o', alpha=0.6)
plt.plot(results_ca["soa"], results_ca["rt_task2"], label="Task A RT (Indep)", marker='x', linestyle='--', alpha=0.6)

plt.xlabel("SOA")
plt.ylabel("RT")
plt.title("PRP Simulation: RT Comparison (Sim Study 3)")
plt.grid(True)
plt.legend()
plt.savefig("../output/plots/RT_p08_zb02_nt100_dt005.png", dpi=300)
plt.show()

# Error Rates
err_a_ba = [1 - a if not np.isnan(a) else np.nan for a in results_ba["acc_task2"]]
err_b_ba = [1 - b if not np.isnan(b) else np.nan for b in results_ba["acc_task1"]]
err_a_ca = [1 - a if not np.isnan(a) else np.nan for a in results_ca["acc_task2"]]
err_c_ca = [1 - b if not np.isnan(b) else np.nan for b in results_ca["acc_task1"]]

#plt.plot(results_ba["soa"], err_b_ba, marker='o', label="Task B Err (Dep)", color="cyan")
plt.plot(results_ba["soa"], err_a_ba, marker='x', linestyle='--', label="Task A Err (Dep)", color="orange")

#plt.plot(results_ca["soa"], err_c_ca, marker='o', label="Task C Err (Indep)", color="darkblue", alpha=0.5)
plt.plot(results_ca["soa"], err_a_ca, marker='x', linestyle='--', label="Task A Err (Indep)", color="red", alpha=0.5)

plt.xlabel("SOA")
plt.ylabel("Error Rate")
plt.title("PRP Simulation: Error Rate Comparison")
plt.grid(True)
plt.legend()
plt.savefig("../output/plots/ER_p08_zb02_nt100_dt005.png", dpi=300)
plt.show()